In [1]:
%env TOKENIZERS_PARALLELISM=false
%env VLLM_USE_V1=0
%env UV_NO_CACHE=1
%env TRANSFORMERS_OFFLINE=1

env: TOKENIZERS_PARALLELISM=false
env: VLLM_USE_V1=0
env: UV_NO_CACHE=1
env: TRANSFORMERS_OFFLINE=1


In [2]:
!uv pip install --no-index --find-links=/kaggle/input/unsloth-2025-10-5 \
unsloth==2025.10.5 \
unsloth-zoo==2025.10.5 \
transformers==4.56.2 \
trl==0.23.0 \
bitsandbytes==0.48.1 \
peft==0.17.1 \
tokenizers==0.22.1 \
huggingface-hub==0.35.3 \
--no-deps

Using Python 3.11.13 environment at: /usr
Resolved 8 packages in 26ms
Prepared 8 packages in 850ms
Uninstalled 4 packages in 533ms
Installed 8 packages in 95ms
 + bitsandbytes==0.48.1
 - huggingface-hub==0.33.1
 + huggingface-hub==0.35.3
 - peft==0.15.2
 + peft==0.17.1
 - tokenizers==0.21.2
 + tokenizers==0.22.1
 - transformers==4.52.4
 + transformers==4.56.2
 + trl==0.23.0
 + unsloth==2025.10.5
 + unsloth-zoo==2025.10.5


In [3]:
!uv pip install --no-index --find-links=/kaggle/input/jigsaw-acrc-wheel loguru
!uv pip install -U --no-deps /kaggle/input/jigsaw-acrc/dist/jigsaw_llm-0.1.0-py3-none-any.whl
!uv pip install -U --no-deps /kaggle/input/jigsaw-acrc/dist/jigsaw-0.1.0-py3-none-any.whl

Using Python 3.11.13 environment at: /usr
Resolved 1 package in 19ms
Prepared 1 package in 9ms
Installed 1 package in 1ms
 + loguru==0.7.3
Using Python 3.11.13 environment at: /usr
Resolved 1 package in 5ms
Prepared 1 package in 7ms
Installed 1 package in 1ms
 + jigsaw-llm==0.1.0 (from file:///kaggle/input/jigsaw-acrc/dist/jigsaw_llm-0.1.0-py3-none-any.whl)
Using Python 3.11.13 environment at: /usr
Resolved 1 package in 5ms
Prepared 1 package in 4ms
Installed 1 package in 1ms
 + jigsaw==0.1.0 (from file:///kaggle/input/jigsaw-acrc/dist/jigsaw-0.1.0-py3-none-any.whl)


In [4]:
%%writefile global_config.py
import os
import sys
import json
import pandas as pd
from pathlib import Path

IS_PRIVATE_RUN = os.getenv('KAGGLE_IS_COMPETITION_RERUN', False)
DEBUG = False
COMPUTE_CV = False
os.environ["COMPUTE_CV"] = str(int(COMPUTE_CV))

if COMPUTE_CV:
    INFERENCE_CSV_PATH = "input/jigsaw-fold/skf_wo_example_4folds.csv"
else:
    INFERENCE_CSV_PATH ="/kaggle/input/jigsaw-agile-community-rules/test.csv"

if IS_PRIVATE_RUN:
    # private test set
    DEBUG = False
    COMPUTE_CV = False
    INFERENCE_CSV_PATH = "/kaggle/input/jigsaw-agile-community-rules/test.csv"

os.environ["INFERENCE_CSV_PATH"] = INFERENCE_CSV_PATH

def print_env() -> None:
    print(f"{IS_PRIVATE_RUN=}")
    print(f"{DEBUG=}")
    print(f"{COMPUTE_CV=}")
    print(f"{INFERENCE_CSV_PATH=}")


PUBLICL_RULES = [
    "No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",
    "No legal advice: Do not offer or request legal advice."
]

Writing global_config.py


In [5]:
import importlib
import global_config
importlib.reload(global_config)
global_config.print_env()

IS_PRIVATE_RUN=False
DEBUG=False
COMPUTE_CV=False
INFERENCE_CSV_PATH='/kaggle/input/jigsaw-agile-community-rules/test.csv'


# Train on Test

In [6]:
!mkdir -p /temp/1
!mkdir -p input/jigsaw-fold
!mkdir -p output

In [7]:
%%writefile input/phi-4.yaml

exp_name: "phi-4"
tracking: true
base_model_name_or_path: "/kaggle/input/phi-unsloth/transformers/4/1"
model_name_or_path: "/kaggle/input/phi-4-unsloth-bnb-4bit"
base_tokenizer_name_or_path: "/kaggle/input/phi-unsloth/transformers/4/1"
tokenizer_name_or_path: "/kaggle/input/phi-4-unsloth-bnb-4bit"
learning_rate: 1e-4
train_batch_size: 4
eval_batch_size: 4
eval_steps_ratio: 1.0
save_steps_ratio: 1.0
num_train_epochs: 1
load_in_4bit: True
load_in_8bit: False
gradient_accumulation_steps: 1

Writing input/phi-4.yaml


In [8]:
%%writefile input/qwen3-14b.yaml

exp_name: "Qwen3-14B"
tracking: true
base_model_name_or_path: "Qwen/Qwen3-14B"
model_name_or_path: "/kaggle/input/qwen3-14b-unsloth-bnb-4bit"
base_tokenizer_name_or_path: "Qwen/Qwen3-14B"
tokenizer_name_or_path: "/kaggle/input/qwen3-14b-unsloth-bnb-4bit"
learning_rate: 2e-4
train_batch_size: 4
eval_batch_size: 4
eval_steps_ratio: 1.0
save_steps_ratio: 1.0
num_train_epochs: 1
load_in_4bit: True
load_in_8bit: False
gradient_accumulation_steps: 1

Writing input/qwen3-14b.yaml


In [9]:
%%writefile input/gemma-2-9b-it.yaml

exp_name: "gemma-2-9b-it"
tracking: true
base_model_name_or_path: "google/gemma-2-9b-it"
model_name_or_path: "/kaggle/input/gemma-2-9b-it-bnb-4bit"
base_tokenizer_name_or_path: "google/gemma-2-9b-it"
tokenizer_name_or_path: "/kaggle/input/gemma-2-9b-it-bnb-4bit"
learning_rate: 1e-4
train_batch_size: 4
eval_batch_size: 4
eval_steps_ratio: 1.0
save_steps_ratio: 1.0
num_train_epochs: 1
load_in_4bit: True
load_in_8bit: False
gradient_accumulation_steps: 1

Writing input/gemma-2-9b-it.yaml


In [10]:
%%writefile input/shieldgemma-9b.yaml

exp_name: "shieldgemma-9b"
tracking: true
base_model_name_or_path: "google/shieldgemma-9b"
model_name_or_path: "/kaggle/input/shieldgemma-9b-bnb-4bit"
base_tokenizer_name_or_path: "google/shieldgemma-9b"
tokenizer_name_or_path: "/kaggle/input/shieldgemma-9b-bnb-4bit"
learning_rate: 8e-5
train_batch_size: 4
eval_batch_size: 4
eval_steps_ratio: 1.0
save_steps_ratio: 1.0
num_train_epochs: 1
load_in_4bit: True
load_in_8bit: False
gradient_accumulation_steps: 1

Writing input/shieldgemma-9b.yaml


In [11]:
%%writefile input/qwen3-8b-guard.yaml

exp_name: "Qwen3Guard-8b"
tracking: true
base_model_name_or_path: "Qwen/Qwen3Guard-Gen-8B"
model_name_or_path: "/kaggle/input/qwen3guard-gen/transformers/8b/1"
base_tokenizer_name_or_path: "Qwen/Qwen3-8B"
tokenizer_name_or_path: "/kaggle/input/qwen3guard-gen/transformers/8b/1"
learning_rate: 2e-4
train_batch_size: 4
eval_batch_size: 4
eval_steps_ratio: 1.0
save_steps_ratio: 1.0
num_train_epochs: 1
load_in_4bit: True
load_in_8bit: False
gradient_accumulation_steps: 1

Writing input/qwen3-8b-guard.yaml


In [12]:
%%writefile train.py

# import os
# os.environ["CUDA_VISIBLE_DEVICES"]="0"

from functools import partial
from pathlib import Path
from typing import Annotated

import datasets
import pandas as pd
import typer
import unsloth
import wandb
from jigsaw_llm.config import BaseConfig
from jigsaw_llm.model import init_model
from jigsaw_llm.prompt.prompt import PromptVer4, PromptVer8, PromptVer9
from jigsaw_llm.train import prepare_dataset
from trl import SFTConfig, SFTTrainer
from trl.trainer.sft_trainer import DataCollatorForLanguageModeling

print("unsloth version: ", unsloth.__version__)


class Config(BaseConfig): ...


INPUT_DIR = Path("input")


def main(
    cfg_path: Annotated[Path, typer.Argument()],
    fold: Annotated[int, typer.Argument()],
    disable_tracking: Annotated[bool, typer.Option("--disable-tracking", is_flag=True)] = False,
) -> None:
    cfg = Config.from_yaml(cfg_path)

    update_dict = {"fold": fold, "tracking": not disable_tracking}
    cfg = cfg.update_from_dict(update_dict)

    if str(cfg.fold) == "99":
        fold=0
    else:
        fold=cfg.fold

    print(f"fold: {fold}")
    typer.echo(f"Using config: {cfg.model_dump_json(indent=2)}")
    output_dir = Path(f"output/{cfg.exp_name}-fold{fold}")

    trainer_args_partial = partial(
        SFTConfig,
        output_dir=str(output_dir),
        overwrite_output_dir=False,
        do_train=True,
        do_eval=False,
        do_predict=False,
        eval_strategy="no",
        prediction_loss_only=True,
        per_device_train_batch_size=cfg.train_batch_size,
        per_device_eval_batch_size=cfg.eval_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        eval_accumulation_steps=1,
        eval_delay=None,
        torch_empty_cache_steps=None,
        learning_rate=cfg.learning_rate,
        weight_decay=0.01,
        max_grad_norm=1.0,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        log_level="error",
        logging_dir=None,
        logging_strategy="steps",
        logging_steps=100,
        save_strategy="steps",
        save_total_limit=1,
        seed=42,
        bf16=False,
        fp16=True,
        bf16_full_eval=False,
        dataloader_drop_last=False,
        remove_unused_columns=False,
        # metric_for_best_model="loss",
        # greater_is_better=False,
        label_smoothing_factor=0.0,
        optim="paged_adamw_8bit",
        group_by_length=False,
        report_to="wandb" if cfg.tracking else "none",
        push_to_hub=False,
        resume_from_checkpoint=None,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        batch_eval_metrics=True,
        max_length=None,
        packing=False,
        padding_free=False,
        eval_packing=False,
        completion_only_loss=False,  # 内部の処理を外だししたのでここではFalse
        activation_offloading=False,
    )

    if "shieldgemma" in cfg_path.name:
        prompt_template = PromptVer8()
    elif "guard" in cfg_path.name:
        prompt_template = PromptVer9()
    else:
        prompt_template = PromptVer4()

    print(prompt_template)

    # drop subreddit duplicates
    train_df = (
        pd.read_csv(INPUT_DIR.joinpath("jigsaw-fold/skf_wo_example_4folds.csv"))
        .drop_duplicates(subset=["rule", "body"], keep="first")
        .reset_index(drop=True)
    )

    trn_df = train_df[train_df["fold"] != cfg.fold].copy()
    trn_df["prompt"] = trn_df.apply(
        lambda row: prompt_template.to_prompt(row),
        axis=1,
    )
    trn_df["completion"] = trn_df["rule_violation"].map(lambda x: "Yes" if bool(x) else "No")
    trn_df = trn_df.sample(frac=1).reset_index(drop=True)  # shuffle

    trn_ds = datasets.Dataset.from_pandas(
        trn_df[["prompt", "completion"]],
    )

    trn_ds = trn_ds.map(
        lambda x: {
            "prompt": [{"role": "user", "content": x["prompt"]}],
            "completion": [{"role": "assistant", "content": x["completion"]}],
        },
    )

    model, tokenizer = init_model(
        cfg.model_name_or_path,
        max_seq_length=1024,
        load_in_4bit=cfg.load_in_4bit,
        load_in_8bit=cfg.load_in_8bit,
    )
    trn_ds = prepare_dataset(trn_ds, tokenizer)
    # trn_ds.save_to_disk("output/llm_dataset")

    total_steps_per_epoch = (len(trn_ds) // cfg.train_batch_size) // cfg.gradient_accumulation_steps
    max_steps = int(total_steps_per_epoch * cfg.num_train_epochs)

    trainer_args = trainer_args_partial(
        max_steps=max_steps,
        save_steps=int(max_steps * cfg.save_steps_ratio),
        eval_steps=int(max_steps * cfg.eval_steps_ratio),
    )

    exp_name = f"{cfg.exp_name}-fold{fold}"
    if cfg.tracking:
        wandb.init(project="kaggle_jigsaw_acrc", name=exp_name)

    trainer = SFTTrainer(
        model=model,
        args=trainer_args,
        train_dataset=trn_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorForLanguageModeling(
            pad_token_id=tokenizer.pad_token_id,  # pyright: ignore
            completion_only_loss=True,
        ),
    )

    trainer.train()
    cfg.save(output_dir)


if __name__ == "__main__":
    typer.run(main)

Writing train.py


In [13]:
%%python
import importlib
import global_config
importlib.reload(global_config)

import pandas as pd
from sklearn.model_selection import StratifiedKFold
from jigsaw.data import convert_example_to_train, count_label, make_rule_body_hash_key, replace_common_label, drop_ambiguous_label

if not global_config.COMPUTE_CV:
    # test側のexampleを展開する
    test_df = convert_example_to_train(
        pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/test.csv"),
        cols=["positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"],
        suffix="test"
    )
    
    # train側のexampleはredditをキーに入れない場合、すべてtrain内のbodyに存在する
    # なのでexample側の情報はいらないはず（example_dfを作ってもhash_keyですべて除外される）
    # testも同じ構造なのかはわからない
    train_df = (
        pd.read_csv("../input/jigsaw-agile-community-rules/train.csv")
        [["row_id", "rule", "subreddit", "body", "rule_violation"]]
    )

    # concat
    df = pd.concat([
        train_df,
        test_df,
    ]).reset_index(drop=True)

    # rule-bodyでハッシュ作成
    df = make_rule_body_hash_key(df)
    # labelをcounting
    df = count_label(df)
    # labelに統一性がないが、labelが不均衡な場合は多数派で置換
    df = replace_common_label(df)
    # labelに統一性がなく、labelが同数であるものは削除
    df = drop_ambiguous_label(df)

    # rule-bodyをsubsetにして重複排除
    df = df.drop_duplicates(subset=["rule_body_hash_key"]).reset_index(drop=True)

    df["fold"]=-1
    df.to_csv("input/jigsaw-fold/skf_wo_example_4folds.csv", index=False)
else:
    df = pd.read_csv("/kaggle/input/jigsaw-fold/skf_wo_example_4folds.csv")
    df.to_csv("input/jigsaw-fold/skf_wo_example_4folds.csv", index=False)

print(pd.read_csv("input/jigsaw-fold/skf_wo_example_4folds.csv"))

      row_id  ... fold
0          0  ...   -1
1          1  ...   -1
2          2  ...   -1
3          3  ...   -1
4          4  ...   -1
...      ...  ...  ...
1864    2024  ...   -1
1865    2025  ...   -1
1866    2026  ...   -1
1867    2027  ...   -1
1868    2028  ...   -1

[1869 rows x 11 columns]


In [14]:
!ls output

In [15]:
%%bash
if [ -n "$KAGGLE_IS_COMPETITION_RERUN" ]; then
    echo "run train"

    CUDA_VISIBLE_DEVICES=0 /usr/local/bin/python train.py input/qwen3-14b.yaml 0 --disable-tracking & 
    CUDA_VISIBLE_DEVICES=1 /usr/local/bin/python train.py input/phi-4.yaml 0 --disable-tracking & 
    wait

    CUDA_VISIBLE_DEVICES=0 /usr/local/bin/python train.py input/gemma-2-9b-it.yaml 0 --disable-tracking &
    CUDA_VISIBLE_DEVICES=1 /usr/local/bin/python train.py input/shieldgemma-9b.yaml 0 --disable-tracking & 
    wait

    CUDA_VISIBLE_DEVICES=0 /usr/local/bin/python train.py input/qwen3-8b-guard.yaml 0 --disable-tracking & 
    wait
fi

# inference

In [16]:
!uv pip install --no-index --find-links=/kaggle/input/vllm-0-10-0 vllm==0.10.0 torchvision
!pip install --no-index --find-links=/kaggle/input/vllm-0-10-0 triton==3.2.0

Using Python 3.11.13 environment at: /usr
Resolved 148 packages in 322ms
Prepared 53 packages in 30.05s
Uninstalled 22 packages in 1.85s
Installed 53 packages in 6.09s
 + astor==0.8.1
 + blake3==1.0.7
 + cbor2==5.7.0
 + compressed-tensors==0.10.2
 + depyf==0.19.0
 + diskcache==5.6.3
 + fastapi-cli==0.0.13
 + fastapi-cloud-cli==0.2.1
 + gguf==0.17.1
 + httptools==0.6.4
 + interegular==0.3.3
 + lark==1.2.2
 + llguidance==0.7.30
 - llvmlite==0.43.0
 + llvmlite==0.44.0
 + lm-format-enforcer==0.10.12
 + mistral-common==1.8.5
 + msgspec==0.19.0
 - numba==0.60.0
 + numba==0.61.2
 - nvidia-cublas-cu12==12.5.3.2
 + nvidia-cublas-cu12==12.6.4.1
 - nvidia-cuda-cupti-cu12==12.5.82
 + nvidia-cuda-cupti-cu12==12.6.80
 - nvidia-cuda-nvrtc-cu12==12.5.82
 + nvidia-cuda-nvrtc-cu12==12.6.77
 - nvidia-cuda-runtime-cu12==12.5.82
 + nvidia-cuda-runtime-cu12==12.6.77
 - nvidia-cudnn-cu12==9.3.0.75
 + nvidia-cudnn-cu12==9.5.1.17
 - nvidia-cufft-cu12==11.2.3.61
 + nvidia-cufft-cu12==11.3.0.4
 + nvidia-cufile-c

In [17]:
%%bash
# enable gemma float16 vllm infernece
sed -i '/^_FLOAT16_NOT_SUPPORTED_MODELS = {$/,/^}$/c\
_FLOAT16_NOT_SUPPORTED_MODELS = {}' /usr/local/lib/python3.11/dist-packages/vllm/config.py

In [18]:
!ls /kaggle/temp/1/

ls: cannot access '/kaggle/temp/1/': No such file or directory


In [19]:
%%bash
echo ${INFERENCE_CSV_PATH}

/kaggle/input/jigsaw-agile-community-rules/test.csv


In [20]:
!ls output

In [21]:
!mkdir output/pred

## Phi-4

In [22]:
%%bash
if [ -n "$KAGGLE_IS_COMPETITION_RERUN" ]; then

    /usr/local/bin/python /kaggle/input/jigsaw-acrc/jigsaw-llm/src/jigsaw_llm/merge.py \
        /kaggle/input/phi-unsloth/transformers/4/1 \
        $(ls -d output/phi-4-fold0/checkpoint-* | sort -V | tail -n 1) \
        -o /kaggle/temp/1/phi-4-fold0 -l -s "1GB" -d "cpu"

else
    echo "pretrained model to temp"

    mkdir -p /kaggle/temp/1/phi-4-fold0
    /usr/local/bin/python /kaggle/input/jigsaw-acrc/jigsaw-llm/src/jigsaw_llm/merge.py \
        /kaggle/input/phi-unsloth/transformers/4/1 \
        /kaggle/input/jigsaw-phi-4-1.0.1-adapter/transformers/1/1/fold0 \
        -o /kaggle/temp/1/phi-4-fold0 -l -s "1GB" -d "cpu"
fi

pretrained model to temp


2025-11-01 03:30:19.072070: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761967819.302107     195 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761967819.369466     195 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 6/6 [00:01<00:00,  5.81it/s]


In [23]:
%%writefile input/vllm_params.yaml

{
    "fold_idx": 0,
    "prompt_version": 4,
    "model": "/kaggle/temp/1/phi-4-fold0",
    "max_model_len": 768,
    "enforce_eager": true,
    "tensor_parallel_size": 2,
    "dtype": "half",
    "gpu_memory_utilization": 0.99,
    "max_num_batched_tokens": 768,
    "enable_lora": false,
    "cpu_offload_gb": 1,
    "swap_space": 1
}

Writing input/vllm_params.yaml


In [24]:
%%bash
/usr/bin/python3 /kaggle/input/jigsaw-acrc/jigsaw-llm/src/jigsaw_llm/inference.py \
${INFERENCE_CSV_PATH} \
output/pred/phi4 input/vllm_params.yaml

INFO 11-01 03:37:35 [__init__.py:235] Automatically detected platform cuda.
Loading vLLM parameters from: {'fold_idx': 0, 'prompt_version': 4, 'model': '/kaggle/temp/1/phi-4-fold0', 'max_model_len': 768, 'enforce_eager': True, 'tensor_parallel_size': 2, 'dtype': 'half', 'gpu_memory_utilization': 0.99, 'max_num_batched_tokens': 768, 'enable_lora': False, 'cpu_offload_gb': 1, 'swap_space': 1}
WARNING 11-01 03:37:54 [config.py:3438] Casting torch.bfloat16 to torch.float16.
INFO 11-01 03:37:54 [config.py:1604] Using max model len 768
WARNING 11-01 03:37:55 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 11-01 03:37:55 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='/kaggle/temp/1/phi-4-fold0', speculative_config=None, tokenizer='/kaggle/temp/1/phi-4-fold0', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_

2025-11-01 03:37:28.982400: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761968249.136677     237 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761968249.152685     237 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
`torch_dtype` is deprecated! Use `dtype` instead!
2025-11-01 03:38:02.604406: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761968282.626695     266 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0

In [25]:
rm -r output/phi-4-fold0

rm: cannot remove 'output/phi-4-fold0': No such file or directory


## Qwen3-14b

In [26]:
%%bash
if [ -n "$KAGGLE_IS_COMPETITION_RERUN" ]; then
    /usr/local/bin/python /kaggle/input/jigsaw-acrc/jigsaw-llm/src/jigsaw_llm/merge.py \
        /kaggle/input/qwen-3/transformers/14b/1 \
        $(ls -d output/Qwen3-14B-fold0/checkpoint-* | sort -V | tail -n 1) \
        -o /kaggle/temp/1/Qwen3-14B-fold0 -l -s "1GB" -d "cpu"
else
    echo "pretrained model to temp"

    mkdir -p /kaggle/temp/1/Qwen3-14B-fold0
    /usr/local/bin/python /kaggle/input/jigsaw-acrc/jigsaw-llm/src/jigsaw_llm/merge.py \
        /kaggle/input/qwen-3/transformers/14b/1 \
        /kaggle/input/jigsaw-qwen3-14b-1.0.1-adapter/transformers/1/3/fold0 \
        -o /kaggle/temp/1/Qwen3-14B-fold0 -l -s "1GB" -d "cpu"
fi

pretrained model to temp


2025-11-01 03:41:45.437356: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761968505.460588     432 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761968505.467395     432 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 8/8 [00:02<00:00,  3.48it/s]


In [27]:
%%writefile input/vllm_params.yaml

{
    "fold_idx": 0,
    "prompt_version": 4,
    "model": "/kaggle/temp/1/Qwen3-14B-fold0",
    "max_model_len": 768,
    "enforce_eager": true,
    "tensor_parallel_size": 2,
    "dtype": "half",
    "gpu_memory_utilization": 0.99,
    "max_num_batched_tokens": 768,
    "enable_lora": false,
    "cpu_offload_gb": 1,
    "swap_space": 1
}

Overwriting input/vllm_params.yaml


In [28]:
%%bash
/usr/bin/python3 /kaggle/input/jigsaw-acrc/jigsaw-llm/src/jigsaw_llm/inference.py \
${INFERENCE_CSV_PATH} \
output/pred/qwen3 input/vllm_params.yaml

INFO 11-01 03:50:33 [__init__.py:235] Automatically detected platform cuda.
Loading vLLM parameters from: {'fold_idx': 0, 'prompt_version': 4, 'model': '/kaggle/temp/1/Qwen3-14B-fold0', 'max_model_len': 768, 'enforce_eager': True, 'tensor_parallel_size': 2, 'dtype': 'half', 'gpu_memory_utilization': 0.99, 'max_num_batched_tokens': 768, 'enable_lora': False, 'cpu_offload_gb': 1, 'swap_space': 1}
WARNING 11-01 03:50:50 [config.py:3438] Casting torch.bfloat16 to torch.float16.
INFO 11-01 03:50:50 [config.py:1604] Using max model len 768
WARNING 11-01 03:50:51 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 11-01 03:50:51 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='/kaggle/temp/1/Qwen3-14B-fold0', speculative_config=None, tokenizer='/kaggle/temp/1/Qwen3-14B-fold0', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}

2025-11-01 03:50:27.857832: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761969027.886740     468 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761969027.894922     468 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
`torch_dtype` is deprecated! Use `dtype` instead!
2025-11-01 03:50:58.608898: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761969058.631798     497 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0

In [29]:
rm -r output/Qwen3-14B-fold0

rm: cannot remove 'output/Qwen3-14B-fold0': No such file or directory


## gemma2-9b-it

In [30]:
%%bash
if [ -n "$KAGGLE_IS_COMPETITION_RERUN" ]; then

    /usr/local/bin/python /kaggle/input/jigsaw-acrc/jigsaw-llm/src/jigsaw_llm/merge.py \
        /kaggle/input/gemma-2/transformers/gemma-2-9b-it/2 \
        $(ls -d output/gemma-2-9b-it-fold0/checkpoint-* | sort -V | tail -n 1) \
        -o /kaggle/temp/1/gemma-2-9b-it-fold0
else
    echo "pretrained model to temp"

    mkdir -p /kaggle/temp/1/gemma-2-9b-it-fold0
    /usr/local/bin/python /kaggle/input/jigsaw-acrc/jigsaw-llm/src/jigsaw_llm/merge.py \
        /kaggle/input/gemma-2/transformers/gemma-2-9b-it/2 \
        /kaggle/input/jigsaw-gemma-2-9b-it-1.0.1-adapter/transformers/1/1/fold0 \
        -o /kaggle/temp/1/gemma-2-9b-it-fold0
fi

pretrained model to temp


2025-11-01 03:54:06.837554: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761969246.861305     609 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761969246.868652     609 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [01:54<00:00, 28.58s/it]


In [31]:
%%writefile input/vllm_params.yaml

{
    "fold_idx": 0,
    "prompt_version": 4,
    "model": "/kaggle/temp/1/gemma-2-9b-it-fold0",
    "max_model_len": 768,
    "enforce_eager": true,
    "tensor_parallel_size": 2,
    "dtype": "half",
    "gpu_memory_utilization": 0.99,
    "max_num_batched_tokens": 768,
    "enable_lora": false
}

Overwriting input/vllm_params.yaml


In [32]:
%%bash
/usr/bin/python3 /kaggle/input/jigsaw-acrc/jigsaw-llm/src/jigsaw_llm/inference.py \
${INFERENCE_CSV_PATH} \
output/pred/gemma2 input/vllm_params.yaml

INFO 11-01 03:57:49 [__init__.py:235] Automatically detected platform cuda.
Loading vLLM parameters from: {'fold_idx': 0, 'prompt_version': 4, 'model': '/kaggle/temp/1/gemma-2-9b-it-fold0', 'max_model_len': 768, 'enforce_eager': True, 'tensor_parallel_size': 2, 'dtype': 'half', 'gpu_memory_utilization': 0.99, 'max_num_batched_tokens': 768, 'enable_lora': False}
WARNING 11-01 03:58:06 [config.py:3438] Casting torch.bfloat16 to torch.float16.
INFO 11-01 03:58:06 [config.py:1604] Using max model len 768
WARNING 11-01 03:58:07 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 11-01 03:58:07 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='/kaggle/temp/1/gemma-2-9b-it-fold0', speculative_config=None, tokenizer='/kaggle/temp/1/gemma-2-9b-it-fold0', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None,

2025-11-01 03:57:44.543165: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761969464.586433     647 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761969464.600403     647 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
`torch_dtype` is deprecated! Use `dtype` instead!
2025-11-01 03:58:14.985722: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761969495.007753     676 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0

In [33]:
!rm -r output/gemma-2-9b-it-fold0

rm: cannot remove 'output/gemma-2-9b-it-fold0': No such file or directory


## shieldgemma fold=0

In [34]:
%%bash

mkdir -p output/shieldgemma-9b-fold0
if [ -n "$KAGGLE_IS_COMPETITION_RERUN" ]; then
    cp -r $(ls -d output/shieldgemma-9b-fold0/checkpoint-* | sort -V | tail -n 1)/* output/shieldgemma-9b-fold0
else
    echo "pretrained model to temp"
    cp -r /kaggle/input/jigsaw-shieldgemma-9b-1.0.1-adapter/transformers/1/3/fold0/* output/shieldgemma-9b-fold0
fi

pretrained model to temp


In [35]:
%%writefile input/vllm_params.yaml

{
    "fold_idx": 0,
    "prompt_version": 8,
    "model": "/kaggle/input/shieldgemma-9b",
    "max_model_len": 768,
    "enforce_eager": true,
    "tensor_parallel_size": 2,
    "dtype": "half",
    "gpu_memory_utilization": 0.99,
    "max_num_batched_tokens": 768,
    "enable_lora": true,
    "lora_path": "output/shieldgemma-9b-fold0"
}

Overwriting input/vllm_params.yaml


In [36]:
%%bash
/usr/bin/python3 /kaggle/input/jigsaw-acrc/jigsaw-llm/src/jigsaw_llm/inference.py \
${INFERENCE_CSV_PATH} \
output/pred/shieldgemma-fold0 input/vllm_params.yaml

INFO 11-01 04:00:47 [__init__.py:235] Automatically detected platform cuda.
Loading vLLM parameters from: {'fold_idx': 0, 'prompt_version': 8, 'model': '/kaggle/input/shieldgemma-9b', 'max_model_len': 768, 'enforce_eager': True, 'tensor_parallel_size': 2, 'dtype': 'half', 'gpu_memory_utilization': 0.99, 'max_num_batched_tokens': 768, 'enable_lora': True, 'lora_path': 'output/shieldgemma-9b-fold0'}
WARNING 11-01 04:01:04 [config.py:3438] Casting torch.bfloat16 to torch.float16.
INFO 11-01 04:01:04 [config.py:1604] Using max model len 768
WARNING 11-01 04:01:05 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 11-01 04:01:05 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='/kaggle/input/shieldgemma-9b', speculative_config=None, tokenizer='/kaggle/input/shieldgemma-9b', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={},

2025-11-01 04:00:41.216556: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761969641.255392     864 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761969641.264324     864 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
`torch_dtype` is deprecated! Use `dtype` instead!
2025-11-01 04:01:12.900882: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761969672.924418     893 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0

In [37]:
!rm -r output/shieldgemma-9b-fold0

# inference only selected rules

In [38]:
%%python
import importlib
import global_config
importlib.reload(global_config)

import pickle
import pandas as pd

df = pd.read_csv(global_config.INFERENCE_CSV_PATH)
print("select rules")

if global_config.IS_PRIVATE_RUN:
    _SELECT_RULE = [r for r in df["rule"].unique() if r not in global_config.PUBLICL_RULES][0]
    SELECT_RULE = [global_config.PUBLICL_RULES[0], _SELECT_RULE]
else:
    SELECT_RULE = [global_config.PUBLICL_RULES[0]]

with open("select_rule.pkl", "wb") as f:
    pickle.dump(SELECT_RULE, f)

df[df["rule"].isin(SELECT_RULE)][["row_id", "body", "rule"]].to_csv("for_select_test.csv", index=False)

select rules


## Qwen3-8b Guard

In [39]:
%%bash

mkdir -p output/Qwen3Guard-8b-fold0
if [ -n "$KAGGLE_IS_COMPETITION_RERUN" ]; then
    cp -r $(ls -d output/Qwen3Guard-8b-fold0/checkpoint-* | sort -V | tail -n 1)/* output/Qwen3Guard-8b-fold0
else
    echo "pretrained model to temp"
    cp -r /kaggle/input/jigsaw-qwen3guard-8b-1.0.1-adapter/transformers/1/1/fold0/* output/Qwen3Guard-8b-fold0
fi

pretrained model to temp


In [40]:
%%writefile input/vllm_params.yaml

{
    "fold_idx": 0,
    "prompt_version": 9,
    "model": "/kaggle/input/qwen3guard-gen/transformers/8b/1",
    "max_model_len": 768,
    "enforce_eager": true,
    "tensor_parallel_size": 2,
    "dtype": "half",
    "gpu_memory_utilization": 0.99,
    "max_num_batched_tokens": 768,
    "enable_lora": true,
    "lora_path": "output/Qwen3Guard-8b-fold0"
}

Overwriting input/vllm_params.yaml


In [41]:
%%bash
/usr/bin/python3 /kaggle/input/jigsaw-acrc/jigsaw-llm/src/jigsaw_llm/inference.py \
for_select_test.csv \
output/pred/Qwen3Guard input/vllm_params.yaml

INFO 11-01 04:04:13 [__init__.py:235] Automatically detected platform cuda.
Loading vLLM parameters from: {'fold_idx': 0, 'prompt_version': 9, 'model': '/kaggle/input/qwen3guard-gen/transformers/8b/1', 'max_model_len': 768, 'enforce_eager': True, 'tensor_parallel_size': 2, 'dtype': 'half', 'gpu_memory_utilization': 0.99, 'max_num_batched_tokens': 768, 'enable_lora': True, 'lora_path': 'output/Qwen3Guard-8b-fold0'}
WARNING 11-01 04:04:28 [config.py:3438] Casting torch.bfloat16 to torch.float16.
INFO 11-01 04:04:28 [config.py:1604] Using max model len 768
WARNING 11-01 04:04:29 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 11-01 04:04:29 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='/kaggle/input/qwen3guard-gen/transformers/8b/1', speculative_config=None, tokenizer='/kaggle/input/qwen3guard-gen/transformers/8b/1', skip_tokenizer_init=False, tokenizer

2025-11-01 04:04:09.124533: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761969849.147463    1128 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761969849.154373    1128 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
`torch_dtype` is deprecated! Use `dtype` instead!
2025-11-01 04:04:36.530865: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761969876.554185    1157 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0

In [42]:
!rm -r output/Qwen3Guard-8b-fold0

# Sub

In [43]:
!ls output/pred

gemma2	phi4  qwen3  Qwen3Guard  shieldgemma-fold0


In [44]:
import numpy as np
import pandas as pd

def ensemble(dfs: list[pd.DataFrame], rule: str, weights: list[float]) -> pd.DataFrame:
    base_df = dfs[0][["row_id", "rule"]].copy()
    base_df = base_df[base_df["rule"]==rule].reset_index(drop=True)

    pred_columns = []
    for i, _df in enumerate(dfs):
        _df = _df[_df["rule"] == rule].reset_index(drop=True)
        assert len(base_df) == len(_df)

        col_name = f"pred_{i}"
        pred_columns.append(col_name)        
        base_df = base_df.merge(
            _df[["row_id", "pred"]].rename(columns={"pred": col_name}),
            on=["row_id"],
            how="left",
        )

    rank_ensemble = np.average(base_df[pred_columns].values, axis=1, weights=weights)

    return pd.DataFrame({
        "row_id": base_df["row_id"],
        "pred": rank_ensemble,
    })


def load_pred_df(path1:str, path2:str) -> pd.DataFrame:
    pred_df = torch.load(path1, weights_only=False)
    pred_df["pred"] = torch.load(path2, weights_only=False)[:, 1]
    pred_df["pred"] = pred_df.groupby("rule")["pred"].rank()
    return pred_df

In [45]:
import importlib
import global_config
importlib.reload(global_config)

import pickle
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

with open("select_rule.pkl", "rb") as f:
    SELECT_RULE = pickle.load(f)

pred_dir = "output/pred"
pred_dfs = [
    load_pred_df(f"{pred_dir}/phi4/fold0_val_df.bin", f"{pred_dir}/phi4/fold0_logits.bin"),
    load_pred_df(f"{pred_dir}/qwen3/fold0_val_df.bin", f"{pred_dir}/qwen3/fold0_logits.bin"),
    load_pred_df(f"{pred_dir}/gemma2/fold0_val_df.bin", f"{pred_dir}/gemma2/fold0_logits.bin"),
    load_pred_df(f"{pred_dir}/shieldgemma-fold0/fold0_val_df.bin", f"{pred_dir}/shieldgemma-fold0/fold0_logits.bin"),
    load_pred_df(f"{pred_dir}/Qwen3Guard/fold0_val_df.bin", f"{pred_dir}/Qwen3Guard/fold0_logits.bin"),
]

sub_df = []

for rule in pred_dfs[0]["rule"].unique():
    if rule in SELECT_RULE:
        sub_df.append(
            ensemble(
                [
                    pred_dfs[0],
                    pred_dfs[1],
                    pred_dfs[2],
                    pred_dfs[3],
                    pred_dfs[4],
                ],
                rule,
                [0.2, 0.4, 0.10, 0.2, 0.10],
            )
        )
    else:
        sub_df.append(
            ensemble(
                [
                    pred_dfs[0],
                    pred_dfs[1],
                    pred_dfs[2],
                    pred_dfs[3],
                ],
                rule,
                weights=[0.3, 0.4, 0.15, 0.15],
            )
        )
        

sub_df = pd.concat(sub_df).rename(columns={"pred":"rule_violation"})
sub_df.to_csv("submission.csv", index=False)
display(sub_df)

,row_id,rule_violation
0,2029,3.2
1,2031,5.8
2,2032,6.1
3,2033,8.3
4,2034,2.5
5,2035,6.9
6,2036,1.3
7,2037,3.0
8,2038,7.9
0,2030,1.0


In [46]:
!find . -not -name 'submission.csv' -delete